# Phase B — TPU Version: End-to-End HBO vs Random Comparison
## Validate that J_topo screening + multi-fidelity selection outperforms random search

**Design:**
- Generate 100 random ThermoNet architectures
- **Random arm**: 20 random configs × 5 epochs
- **HBO arm**: J_topo screen 100→20, multi-fidelity (5→50 epochs)
- **Metric**: Compare val_loss of top-5 candidates at 50 epochs

**Expected**: If J_topo → E_floor correlation is real, HBO should win.


## Step 1: TPU Setup

In [ ]:
# TPU setup
import os
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl

try:
    DEVICE = xm.xla_device()
    test_tensor = torch.zeros((2, 3), device=DEVICE)
    print('TPU device:', DEVICE)
    print('TPU runtime: OK')
except Exception as e:
    raise RuntimeError(f'TPU not accessible: {e}. '
        'Notebook Settings → Accelerator → TPU → Save.')


## Step 2: Clone

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret('gh_token')
repo_url = f'https://{gh_token}@github.com/xliu203/ThermoRG-NN.git'
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email 'xliu203@asu.edu'
!git config --global user.name 'Leo Liu'


## Step 3: Imports

In [ ]:
import sys, math, time, json, random
sys.path.insert(0, '/kaggle/working/ThermoRG-NN')
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/experiments/phase_b')

import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from scipy.optimize import curve_fit
from scipy.linalg import svd

import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

from thermorg import compute_J_topo
from thermorg.scaling import fit_scaling_law
from scipy.stats import spearmanr

print(f'PyTorch: {torch.__version__}')
print(f'TPU: {xm.xla_device()}')


## Step 4: Architecture Space + Network

In [ ]:
# ─── Architecture Encoding ────────────────────────────────────────────
# ThermoNet: 4-dim space (width, depth, norm, skip)

# Width options (base_ch)
WIDTHS = [16, 24, 32, 48, 64, 96]
# Depth options (number of conv layers per block)
DEPTHS = [3, 4, 5, 6]
# Norm types
NORMS = ['none', 'batchnorm', 'layernorm']
# Skip connection
SKIPS = [False, True]

# Generate 100 random architectures
random.seed(42)
CANDIDATES = []
for _ in range(100):
    w = random.choice(WIDTHS)
    d = random.choice(DEPTHS)
    n = random.choice(NORMS)
    s = random.choice(SKIPS)
    CANDIDATES.append({'width': w, 'depth': d, 'norm': n, 'skip': s})

print(f'Generated {len(CANDIDATES)} candidate architectures')
print(f'Norm distribution: none={sum(1 for c in CANDIDATES if c["norm"]=="none")}, '
      f'bn={sum(1 for c in CANDIDATES if c["norm"]=="batchnorm")}, '
      f'ln={sum(1 for c in CANDIDATES if c["norm"]=="layernorm")}')


In [ ]:
# ─── Network Definition ──────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm_type='none', use_skip=False, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False)
        if norm_type == 'batchnorm':
            self.norm = nn.BatchNorm2d(out_ch)
        elif norm_type == 'layernorm':
            self.norm = nn.LayerNorm([out_ch, 32//stride, 32//stride])
        else:
            self.norm = nn.Identity()
        self.act = nn.GELU()
        self.use_skip = use_skip and (in_ch == out_ch) and (stride == 1)
        self.skip = nn.Identity() if self.use_skip else None

    def forward(self, x):
        out = self.norm(self.conv(x))
        out = self.act(out)
        if self.skip is not None:
            out = out + self.skip(x)
        return out


class ThermoNet(nn.Module):
    def __init__(self, base_ch=64, depth=3, norm_type='none', use_skip=False, n_classes=10):
        super().__init__()
        channels = [3] + [base_ch] * depth
        self.blocks = nn.ModuleList()
        for i in range(depth):
            self.blocks.append(ConvBlock(
                channels[i], channels[i+1],
                norm_type=norm_type,
                use_skip=(i > 0 and use_skip),
                stride=1
            ))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels[-1], n_classes)

    def get_conv_weights(self):
        return [m.weight.data for m in self.modules() if isinstance(m, nn.Conv2d)]

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


## Step 5: J_topo + DataLoaders

In [ ]:
# Compute J_topo for all 100 candidates
print('Computing J_topo for all candidates...')
t0 = time.time()
for cfg in CANDIDATES:
    model = ThermoNet(base_ch=cfg['width'], depth=cfg['depth'], norm_type=cfg['norm'], use_skip=cfg['skip'])
    weights = model.get_conv_weights()
    cfg['J_topo'] = compute_J_topo(weights)
    del model

print(f'J_topo computed in {time.time()-t0:.1f}s')
J_vals = [c['J_topo'] for c in CANDIDATES]
print(f'J_topo range: [{min(J_vals):.4f}, {max(J_vals):.4f}], mean={np.mean(J_vals):.4f}')

# DataLoaders
tfm_train = T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),
                       T.ToTensor(),T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])])
tfm_val  = T.Compose([T.ToTensor(),T.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])])
train_ds = CIFAR10(root='./data', train=True,  transform=tfm_train, download=True)
val_ds   = CIFAR10(root='./data', train=False, transform=tfm_val,  download=True)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')


## Step 6: Training Functions

In [ ]:
BATCH_SIZE = 256
LR = 0.01
WD = 5e-4
MOMENTUM = 0.9

def get_loaders(batch_size=BATCH_SIZE):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader


def train_one(model, train_loader, val_loader, epochs, lr=LR, wd=WD, momentum=MOMENTUM):
    """TPU training: returns best_val_loss, val_acc at best epoch."""
    model = model.to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    best_loss, best_acc = float('inf'), 0.0
    best_ep = 0
    
    for epoch in range(epochs):
        model.train()
        mp = pl.MpDeviceLoader(train_loader, DEVICE)
        for X, y in mp:
            opt.zero_grad()
            loss = crit(model(X), y)
            loss.backward()
            xm.optimizer_step(opt, barrier=True)
        sched.step()
        
        model.eval()
        mp_v = pl.MpDeviceLoader(val_loader, DEVICE)
        loss_sum, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X, y in mp_v:
                out = model(X)
                loss_sum += crit(out, y).item() * X.size(0)
                correct += (out.argmax(1)==y).sum().item()
                total += X.size(0)
        val_loss = loss_sum/total
        val_acc = correct/total
        if val_loss < best_loss:
            best_loss, best_acc, best_ep = val_loss, val_acc, epoch+1
    
    del model
    return {'best_val_loss': best_loss, 'best_val_acc': best_acc, 'best_epoch': best_ep}


## Step 7: Stage 1 — L1 Screening (Random vs HBO)

In [ ]:
OUT_DIR = Path('/kaggle/working/phase_b_tpu_results')
OUT_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

train_loader, val_loader = get_loaders()

# ── Random arm: 20 random configs ──
print('\n' + '='*60)
print('STAGE 1a: Random arm — 20 random configs × 5 epochs')
print('='*60)
random.seed(12345)
RANDOM_CFGS = random.sample(CANDIDATES, 20)
RANDOM_RESULTS = []

t0 = time.time()
for i, cfg in enumerate(RANDOM_CFGS):
    ckpt = CKPT_DIR / f'rand_s1_{i}.pt'
    if ckpt.exists():
        with open(OUT_DIR / f'rand_s1_{i}.json') as f:
            res = json.load(f)
        RANDOM_RESULTS.append(res)
        print(f'  [{i+1}/20] {cfg} [SKIP] loss={res["best_val_loss"]:.4f}')
        continue
    print(f'  [{i+1}/20] Training {cfg}...')
    torch.manual_seed(42 + i)
    model = ThermoNet(base_ch=cfg['width'], depth=cfg['depth'], norm_type=cfg['norm'], use_skip=cfg['skip'])
    res = train_one(model, train_loader, val_loader, epochs=5)
    res['config'] = cfg
    RANDOM_RESULTS.append(res)
    with open(OUT_DIR / f'rand_s1_{i}.json', 'w') as f:
        json.dump(res, f)
    print(f'      → loss={res["best_val_loss"]:.4f} acc={res["best_val_acc"]:.4f}')

rand_l1_time = time.time() - t0
print(f'\nRandom L1 done in {rand_l1_time/60:.1f} min')
print(f'Random top-5 by L1:')
rand_sorted = sorted(RANDOM_RESULTS, key=lambda x: x['best_val_loss'])
for r in rand_sorted[:5]:
    print(f'  loss={r["best_val_loss"]:.4f} {r["config"]}')


In [ ]:
# ── HBO arm: J_topo screen 100→20 ──
print('\n' + '='*60)
print('STAGE 1b: HBO arm — J_topo top-20 × 5 epochs')
print('='*60)

# Sort by J_topo descending (higher = better topology)
sorted_by_J = sorted(CANDIDATES, key=lambda x: x['J_topo'], reverse=True)
HBO_CFGS = sorted_by_J[:20]  # top 20 by J_topo

print(f'J_topo range in HBO set: [{HBO_CFGS[-1]["J_topo"]:.4f}, {HBO_CFGS[0]["J_topo"]:.4f}]')
HBO_RESULTS = []

t0 = time.time()
for i, cfg in enumerate(HBO_CFGS):
    ckpt = CKPT_DIR / f'hbo_s1_{i}.pt'
    if ckpt.exists():
        with open(OUT_DIR / f'hbo_s1_{i}.json') as f:
            res = json.load(f)
        HBO_RESULTS.append(res)
        print(f'  [{i+1}/20] {cfg} [SKIP] loss={res["best_val_loss"]:.4f}')
        continue
    print(f'  [{i+1}/20] Training {cfg}...')
    torch.manual_seed(42 + i + 1000)
    model = ThermoNet(base_ch=cfg['width'], depth=cfg['depth'], norm_type=cfg['norm'], use_skip=cfg['skip'])
    res = train_one(model, train_loader, val_loader, epochs=5)
    res['config'] = cfg
    HBO_RESULTS.append(res)
    with open(OUT_DIR / f'hbo_s1_{i}.json', 'w') as f:
        json.dump(res, f)
    print(f'      → loss={res["best_val_loss"]:.4f} acc={res["best_val_acc"]:.4f}')

hbo_l1_time = time.time() - t0
print(f'\nHBO L1 done in {hbo_l1_time/60:.1f} min')
print(f'HBO top-5 by L1:')
hbo_sorted = sorted(HBO_RESULTS, key=lambda x: x['best_val_loss'])
for r in hbo_sorted[:5]:
    print(f'  loss={r["best_val_loss"]:.4f} J_topo={r["config"]["J_topo"]:.4f} {r["config"]}')


## Step 8: Stage 1 Analysis

In [ ]:
# Compare L1 results
print('\n' + '='*60)
print('STAGE 1 ANALYSIS: L1 (5-epoch) Comparison')
print('='*60)

rand_losses = [r['best_val_loss'] for r in RANDOM_RESULTS]
hbo_losses  = [r['best_val_loss'] for r in HBO_RESULTS]

print(f'Random:  mean={np.mean(rand_losses):.4f}, std={np.std(rand_losses):.4f}, min={min(rand_losses):.4f}')
print(f'HBO:     mean={np.mean(hbo_losses):.4f}, std={np.std(hbo_losses):.4f}, min={min(hbo_losses):.4f}')
print(f'Delta (HBO - Random): mean={np.mean(hbo_losses)-np.mean(rand_losses):.4f}')

# Top-K comparison
for k in [1, 3, 5]:
    rk = sorted(rand_losses)[k-1] if k <= len(rand_losses) else float('inf')
    hk = sorted(hbo_losses)[k-1]  if k <= len(hbo_losses)  else float('inf')
    print(f'Top-{k}: Random={rk:.4f}, HBO={hk:.4f}, Δ={hk-rk:.4f}')

# Spearman correlation: J_topo vs L1 loss (within HBO arm)
J_vals_hbo = [r['config']['J_topo'] for r in HBO_RESULTS]
losses_hbo = [r['best_val_loss'] for r in HBO_RESULTS]
rho, p = spearmanr(J_vals_hbo, losses_hbo)
print(f'\nSpearman (J_topo vs L1 loss, HBO arm): r={rho:.3f}, p={p:.4f}')


## Step 9: Stage 2 — GP Surrogate + Selection

In [ ]:
# Simple GP: model E_floor from [J_topo, L1_loss]
# Use sklearn GaussianProcessRegressor for simplicity
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

# Features: [J_topo, L1_loss]
# Target: final val_loss at 50 epochs

print('\n' + '='*60)
print('STAGE 2: GP Surrogate — train on L1, predict L2')
print('='*60)

# Train GP on all L1 data (both arms combined)
X_all = np.array([[r['config']['J_topo'], r['best_val_loss']] for r in RANDOM_RESULTS + HBO_RESULTS])
y_all = np.array([r['best_val_loss'] for r in RANDOM_RESULTS + HBO_RESULTS])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

kernel = ConstantKernel(0.1) * RBF(length_scale=[1.0, 1.0]) + WhiteKernel(0.1)
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.01, normalize_y=True, random_state=42)
gp.fit(X_scaled, y_all)

print(f'GP trained on {len(X_all)} L1 observations')
print(f'GP log-marginal-likelihood: {gp.log_marginal_likelihood_value_:.3f}')


## Step 10: Stage 2 — L2 Training (Top-5 × 50 epochs)

In [ ]:
# ── Random top-5 at L2 (50 epochs) ──
print('\n' + '='*60)
print('STAGE 2: Random arm top-5 × 50 epochs')
print('='*60)
RANDOM_TOP5 = rand_sorted[:5]
RANDOM_L2 = []

t0 = time.time()
for i, res in enumerate(RANDOM_TOP5):
    cfg = res['config']
    ckpt = CKPT_DIR / f'rand_s2_{i}.pt'
    if ckpt.exists():
        with open(OUT_DIR / f'rand_s2_{i}.json') as f:
            l2_res = json.load(f)
        RANDOM_L2.append(l2_res)
        print(f'  [{i+1}/5] [SKIP] loss={l2_res["best_val_loss"]:.4f}')
        continue
    print(f'  [{i+1}/5] Training {cfg}...')
    torch.manual_seed(42 + i)
    model = ThermoNet(base_ch=cfg['width'], depth=cfg['depth'], norm_type=cfg['norm'], use_skip=cfg['skip'])
    l2_res = train_one(model, train_loader, val_loader, epochs=50)
    l2_res['config'] = cfg
    RANDOM_L2.append(l2_res)
    with open(OUT_DIR / f'rand_s2_{i}.json', 'w') as f:
        json.dump(l2_res, f)
    print(f'      → loss={l2_res["best_val_loss"]:.4f} acc={l2_res["best_val_acc"]:.4f}')

rand_l2_time = time.time() - t0
print(f'\nRandom L2 done in {rand_l2_time/60:.1f} min')


In [ ]:
# ── HBO: GP-guided top-5 at L2 (50 epochs) ──
print('\n' + '='*60)
print('STAGE 2: HBO arm — GP top-5 × 50 epochs')
print('='*60)

# Use GP to predict best from HBO candidates
X_hbo = np.array([[r['config']['J_topo'], r['best_val_loss']] for r in HBO_RESULTS])
X_hbo_scaled = scaler.transform(X_hbo)
pred_mean, pred_std = gp.predict(X_hbo_scaled, return_std=True)

# Select by predicted loss (lower is better)
hbo_with_pred = list(zip(HBO_RESULTS, pred_mean, pred_std))
hbo_with_pred.sort(key=lambda x: x[1])  # sort by predicted mean

HBO_TOP5 = [r for r, _, _ in hbo_with_pred[:5]]
HBO_L2 = []

t0 = time.time()
for i, res in enumerate(HBO_TOP5):
    cfg = res['config']
    # Find index in HBO_RESULTS
    orig_idx = HBO_RESULTS.index(res)
    ckpt = CKPT_DIR / f'hbo_s2_{i}.pt'
    if ckpt.exists():
        with open(OUT_DIR / f'hbo_s2_{i}.json') as f:
            l2_res = json.load(f)
        HBO_L2.append(l2_res)
        print(f'  [{i+1}/5] [SKIP] loss={l2_res["best_val_loss"]:.4f}')
        continue
    print(f'  [{i+1}/5] Training {cfg} (GP idx={orig_idx})...')
    torch.manual_seed(42 + orig_idx + 1000)
    model = ThermoNet(base_ch=cfg['width'], depth=cfg['depth'], norm_type=cfg['norm'], use_skip=cfg['skip'])
    l2_res = train_one(model, train_loader, val_loader, epochs=50)
    l2_res['config'] = cfg
    HBO_L2.append(l2_res)
    with open(OUT_DIR / f'hbo_s2_{i}.json', 'w') as f:
        json.dump(l2_res, f)
    print(f'      → loss={l2_res["best_val_loss"]:.4f} acc={l2_res["best_val_acc"]:.4f}')

hbo_l2_time = time.time() - t0
print(f'\nHBO L2 done in {hbo_l2_time/60:.1f} min')


## Step 11: Final Results + Comparison

In [ ]:
print('\n' + '='*60)
print('FINAL RESULTS: HBO vs Random (50-epoch) Comparison')
print('='*60)

rand_l2_losses = [r['best_val_loss'] for r in RANDOM_L2]
hbo_l2_losses  = [r['best_val_loss'] for r in HBO_L2]

print(f'\nRandom top-5 (L2, 50 epochs):')
for r in sorted(RANDOM_L2, key=lambda x: x['best_val_loss']):
    print(f'  loss={r["best_val_loss"]:.4f} J_topo={r["config"]["J_topo"]:.4f} {r["config"]}')

print(f'\nHBO top-5 (L2, 50 epochs):')
for r in sorted(HBO_L2, key=lambda x: x['best_val_loss']):
    print(f'  loss={r["best_val_loss"]:.4f} J_topo={r["config"]["J_topo"]:.4f} {r["config"]}')

print(f'\nSummary:')
print(f'  Random best:  {min(rand_l2_losses):.4f} (mean: {np.mean(rand_l2_losses):.4f})')
print(f'  HBO best:    {min(hbo_l2_losses):.4f} (mean: {np.mean(hbo_l2_losses):.4f})')
print(f'  HBO improvement: {min(rand_l2_losses) - min(hbo_l2_losses):.4f}')

print(f'\nStage 1 (5-epoch):')
print(f'  Random mean: {np.mean(rand_losses):.4f}')
print(f'  HBO mean:    {np.mean(hbo_losses):.4f}')

print(f'\nL1→L2 correlation (for GP quality):')
l1_vals = [r['best_val_loss'] for r in RANDOM_L2]
l2_vals = [r['best_val_loss'] for r in RANDOM_L2]
rho, p = spearmanr(l1_vals, l2_vals)
print(f'  Spearman r(L1, L2) = {rho:.3f} (p={p:.4f})')


## Step 12: Save + Push

In [ ]:
# Save final results
final_results = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'random_l1': [{'config': r['config'], 'loss': r['best_val_loss']} for r in RANDOM_RESULTS],
    'hbo_l1':    [{'config': r['config'], 'loss': r['best_val_loss']} for r in HBO_RESULTS],
    'random_top5_l2': [{'config': r['config'], 'loss': r['best_val_loss']} for r in RANDOM_L2],
    'hbo_top5_l2':    [{'config': r['config'], 'loss': r['best_val_loss']} for r in HBO_L2],
    'random_l2_min': float(min(rand_l2_losses)),
    'hbo_l2_min':     float(min(hbo_l2_losses)),
    'hbo_improvement': float(min(rand_l2_losses) - min(hbo_l2_losses)),
}
with open(OUT_DIR / 'phase_b_final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f'\nResults saved to {OUT_DIR}')

# Push to GitHub
cmds = [
    f'git add {OUT_DIR}/*.json 2>/dev/null || true',
    f'git add {CKPT_DIR}/*.pt 2>/dev/null || true',
    "git add experiments/phase_b/notebooks/*.ipynb 2>/dev/null || true",
    "git commit -m 'Data: Phase B HBO vs Random comparison' || true",
    'git push origin develop 2>&1 || true',
]
for cmd in cmds:
    !$cmd
print('Push done')
